# Part 3 - Machine Learning: Task 3.1 Features, split and baseline

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("appointments_clean.csv")
print("Loaded:", df.shape)

X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print("X shape:", X.shape)
print("Feature columns:", X.columns.tolist())
print("Missing values in X:", X.isnull().sum().sum())

Loaded: (990, 16)
X shape: (990, 11)
Feature columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Missing values in X: 0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", len(X_train))
print("Test size:", len(X_test))
print("No-shows in test set:", y_test.sum())
print("Overall no-show share:", round(y.mean() * 100, 1), "%")

baseline_acc = (y_test == 0).mean()
print("Baseline accuracy (always Attended):", round(baseline_acc, 4))
print("Correct predictions:", (y_test == 0).sum(), "of", len(y_test))

Training size: 792
Test size: 198
No-shows in test set: 49
Overall no-show share: 22.0 %
Baseline accuracy (always Attended): 0.7525
Correct predictions: 149 of 198


## Task 3.2 - Logistic regression

In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1       :", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 3))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.788


Precision: 0.63
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789
Confusion matrix:
[[139  10]
 [ 32  17]]


In [10]:
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coef": model.coef_[0],
    "scaler_scale": scaler.scale_,
})
print(coef_table.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
coef_r = model.coef_[0][i]
scale_r = scaler.scale_[i]

print()
print("reminder_sent coefficient:", round(coef_r, 3))
print("reminder_sent scale (SD) :", round(scale_r, 3))
print("exp(coef)         [per 1 SD, scaled feature]:", round(np.exp(coef_r), 2))
print("exp(coef / scale) [real No -> Yes change]   :", round(np.exp(coef_r / scale_r), 2))
print("Odds reduction for reminder (No -> Yes):", round((1 - np.exp(coef_r / scale_r)) * 100), "%")

                feature   coef  scaler_scale
                    age -0.254        16.656
            distance_km  0.253         6.052
                fee_inr  0.453       184.254
              lead_days  0.614         8.614
         prior_no_shows  0.650         0.969
          reminder_sent -0.581         0.495
    booking_channel_App -0.047         0.486
  booking_channel_Phone  0.216         0.481
booking_channel_Walk-In -0.186         0.435
visit_type_Consultation  0.016         0.493
   visit_type_Follow-up -0.016         0.493

reminder_sent coefficient: -0.581
reminder_sent scale (SD) : 0.495
exp(coef)         [per 1 SD, scaled feature]: 0.56
exp(coef / scale) [real No -> Yes change]   : 0.31
Odds reduction for reminder (No -> Yes): 69 %


(a) The scaled coefficient gives an odds ratio of 0.56 per one standard deviation of reminder_sent, but reminder_sent only has a standard deviation of about 0.5, so the real No->Yes change (a full 1-unit step) gives 0.31.

(b) In this model, holding the other features constant, bookings with a reminder have about 69% lower odds of a no-show. This is an association in observational data and does not prove that reminders cause fewer no-shows.

## Task 3.3 - Decision tree and overfitting

In [11]:
from sklearn.tree import DecisionTreeClassifier

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

tree_d3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_d3.fit(X_train, y_train)

print("Full tree   - depth:", tree_full.get_depth())
print("Full tree   - train accuracy:", round(tree_full.score(X_train, y_train), 3))
print("Full tree   - test accuracy :", round(tree_full.score(X_test, y_test), 3))
print()
print("Depth-3 tree - train accuracy:", round(tree_d3.score(X_train, y_train), 3))
print("Depth-3 tree - test accuracy :", round(tree_d3.score(X_test, y_test), 3))
print()
print("Baseline (always Attended):", round((y_test == 0).mean(), 4))

Full tree   - depth:

 15
Full tree   - train accuracy: 1.0
Full tree   - test accuracy : 0.672

Depth-3 tree - train accuracy: 0.822
Depth-3 tree - test accuracy : 0.778

Baseline (always Attended): 0.7525


The full tree scores 1.00 on the training data but only about 0.67 on the test data, which is below the 0.7525 baseline, so it memorised noise (overfitting); the depth-3 tree has a much smaller gap (about 0.82 vs 0.78) and generalises better.

## Task 3.4 - Random forest and comparison

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("RF train accuracy:", round(rf.score(X_train, y_train), 3))
print("RF test accuracy :", round(accuracy_score(y_test, rf_pred), 3))
print("RF recall        :", round(recall_score(y_test, rf_pred), 3))
print("RF ROC-AUC       :", round(roc_auc_score(y_test, rf_prob), 3))

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print()
print("Top 3 feature importances:")
print(importances.head(3).round(3))

RF train accuracy: 1.0
RF test accuracy : 0.747
RF recall        : 0.306
RF ROC-AUC       : 0.706

Top 3 feature importances:
lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64


In [13]:
lr_pred = model.predict(X_test_s)
lr_prob = model.predict_proba(X_test_s)[:, 1]
d3_pred = tree_d3.predict(X_test)
d3_prob = tree_d3.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, lr_pred),
                 accuracy_score(y_test, d3_pred),
                 accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, lr_pred),
               recall_score(y_test, d3_pred),
               recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, lr_prob),
                roc_auc_score(y_test, d3_prob),
                roc_auc_score(y_test, rf_prob)],
}).round(3)

print(comparison.to_string(index=False))

                  model  accuracy  recall  roc_auc
    Logistic regression     0.788   0.347    0.789
Decision tree (depth 3)     0.778   0.224    0.700
          Random forest     0.747   0.306    0.706


On the test set, logistic regression looks best: it has the highest ROC-AUC and the highest accuracy and recall of the three models. The random forest memorised the training data (training accuracy 1.00) but does not beat the simpler model on new data. This table is for reporting only; the formal model choice is made in Task 3.5 using cross-validation on the training set.

## Task 3.5 - Tune, choose, then test once

In [14]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict

param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)

print("best_params_:", grid.best_params_)
print("best_score_ (CV ROC-AUC):", round(grid.best_score_, 3))

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (CV ROC-AUC): 0.767


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])

cv_scores = {
    "Logistic regression": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned forest": grid.best_score_,
}

for name, score in cv_scores.items():
    print(f"{name:25s} CV ROC-AUC: {score:.3f}")

winner = max(cv_scores, key=cv_scores.get)
print()
print("Winner (highest CV ROC-AUC):", winner)

Logistic regression       CV ROC-AUC: 0.774
Decision tree (depth 3)   CV ROC-AUC: 0.678
Random forest             CV ROC-AUC: 0.747
Tuned forest              CV ROC-AUC: 0.767

Winner (highest CV ROC-AUC): Logistic regression


In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5,
                        method="predict_proba")[:, 1]

for t in [0.5, 0.35]:
    pred = (oof >= t).astype(int)
    print(f"Threshold {t}: precision={precision_score(y_train, pred):.2f}, "
          f"recall={recall_score(y_train, pred):.2f}, "
          f"F1={f1_score(y_train, pred):.2f}, flagged={pred.sum()}")

Threshold 0.5: precision=0.58, recall=0.25, F1=0.35, flagged=73
Threshold 0.35: precision=0.51, recall=0.47, F1=0.49, flagged=154


In [17]:
threshold = 0.35

lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= threshold).astype(int)

print("TEST SET (evaluated once) - logistic regression, threshold", threshold)
print("Precision:", round(precision_score(y_test, flag), 2))
print("Recall   :", round(recall_score(y_test, flag), 2))
print("Flagged  :", flag.sum())
print("ROC-AUC  :", round(roc_auc_score(y_test, probs), 2))

scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": flag,
})
scores.to_csv("test_risk_scores.csv", index=False)

check = pd.read_csv("test_risk_scores.csv")
print()
print("Saved shape:", check.shape)
print("Columns:", check.columns.tolist())
print("Rows with flag = 1:", check["flag"].sum())
print(check.head())

TEST SET (evaluated once) - logistic regression, threshold 0.35
Precision: 0.57
Recall   : 0.57
Flagged  : 49
ROC-AUC  : 0.79

Saved shape: (198, 4)
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Rows with flag = 1: 49
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0


(a) Logistic regression won the 5-fold CV ROC-AUC on the training set (about 0.77), ahead of the tuned forest, the random forest and the depth-3 tree.
(b) I chose 0.35 because on out-of-fold training predictions it gave a higher F1 (about 0.49 vs 0.35) and much higher recall (about 0.47 vs 0.25) than 0.5; a missed warning costs an empty slot, while an extra call costs about a minute.
(c) A clinic would choose the threshold by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot.